# Health Insurance Classification Model
## Anova Insurance - Premium Pricing Decision Support

**Objective:** Develop a predictive model to classify individuals as 'Healthy' or 'Unhealthy' 
based on health data to assist in insurance policy premium pricing.

In [1]:
pip install pandas scikit-learn matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# ============================================================================
# IMPORT REQUIRED LIBRARIES FOR DATA ANALYSIS & MACHINE LEARNING
# ============================================================================

# pandas - Data manipulation and analysis
# Required for: Loading CSV files, data cleaning, exploring DataFrames, handling missing values
import pandas as pd

# numpy - Numerical computing library
# Required for: Array operations, mathematical calculations, numerical data manipulation
import numpy as np

# matplotlib - Plotting and visualization library (base plotting)
# Required for: Creating static plots, histograms, line charts, scatter plots
import matplotlib.pyplot as plt

# seaborn - Statistical data visualization built on matplotlib
# Required for: Advanced visualizations like heatmaps, distribution plots, correlation matrices
import seaborn as sns

# sklearn preprocessing - Data preprocessing tools
# StandardScaler: Standardizes features by removing mean and scaling to unit variance
# Required for: Normalizing numerical features (important for models like SVM, KNN, Neural Networks)
from sklearn.preprocessing import StandardScaler

# sklearn model_selection - Tools for splitting and validating models
# train_test_split: Splits data into training and testing sets
# Required for: Evaluating model performance on unseen data
from sklearn.model_selection import train_test_split

# sklearn metrics - Evaluation metrics for classification models
# classification_report: Detailed metrics (precision, recall, F1-score)
# confusion_matrix: Shows True Positives, False Positives, etc.
# roc_auc_score: Measures model's ability to distinguish between classes
# Required for: Assessing model performance and comparing different models
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# warnings - Suppress warning messages
# Required for: Clean output by hiding non-critical warnings that may clutter notebook results
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURE VISUALIZATION SETTINGS
# ============================================================================

# Set seaborn style for better-looking plots
sns.set_style("whitegrid")

# Set default figure size for all matplotlib plots
plt.rcParams['figure.figsize'] = (12, 6)

In [3]:
# Load the dataset
df = pd.read_csv('../data/Healthcare_Dataset_Preprocessed.csv')

# Display basic information
print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())

Dataset Shape: (9549, 23)

First few rows:
    Age   BMI  Blood_Pressure  Cholesterol  Glucose_Level  Heart_Rate  \
0   2.0  26.0           111.0        198.0           99.0        72.0   
1   8.0  24.0           121.0        199.0          103.0        75.0   
2  81.0  27.0           147.0        203.0          100.0        74.0   
3  25.0  21.0           150.0        199.0          102.0        70.0   
4  24.0  26.0           146.0        202.0           99.0        76.0   

   Sleep_Hours  Exercise_Hours  Water_Intake  Stress_Level  ...  Diet  \
0          4.0             1.0           5.0           5.0  ...     1   
1          2.0             1.0           2.0           9.0  ...     1   
2         10.0            -0.0           5.0           1.0  ...     2   
3          7.0             3.0           3.0           3.0  ...     1   
4         10.0             2.0           5.0           1.0  ...     2   

   MentalHealth  PhysicalActivity  MedicalHistory  Allergies  Diet_Type_Vegan  

## 1. Data Exploration

Let's explore the target variable and understand class distribution

In [ ]:
# ============================================================================
# STEP 1: EXPLORE TARGET VARIABLE DISTRIBUTION
# ============================================================================
# Why: Understanding class balance (Healthy vs Unhealthy) is crucial for model performance
# If severely imbalanced, we may need special techniques (SMOTE, class weights, etc.)

print("=" * 70)
print("TARGET VARIABLE DISTRIBUTION")
print("=" * 70)

# Count of each class (0=Healthy, 1=Unhealthy)
target_counts = df['Target'].value_counts()
print("\nClass Distribution:")
print(target_counts)

# Calculate percentage distribution
target_percentage = (df['Target'].value_counts() / len(df)) * 100
print("\nPercentage Distribution:")
print(target_percentage)

# Visualize class distribution
plt.figure(figsize=(8, 5))
sns.countplot(x='Target', data=df, palette='Set2')
plt.title('Distribution of Healthy vs Unhealthy Individuals', fontsize=14, fontweight='bold')
plt.xlabel('Target (0=Healthy, 1=Unhealthy)', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks([0, 1], ['Healthy (0)', 'Unhealthy (1)'])
plt.show()

print("\n✓ Class balance analyzed. Imbalance ratio:", target_counts[1] / target_counts[0])

## 2. Data Preprocessing for Decision Tree

In [ ]:
# ============================================================================
# STEP 2: DATA PREPROCESSING
# ============================================================================
# Why: Decision trees require clean data. We need to handle missing values and categoricals

print("=" * 70)
print("DATA PREPROCESSING")
print("=" * 70)

# Create a copy to work with
df_processed = df.copy()

# Handle missing values
print("\nMissing values before imputation:")
print(df_processed.isnull().sum())

# Strategy: Fill numerical missing values with median (robust to outliers)
# Why median: Less sensitive to extreme values than mean
numerical_cols = df_processed.select_dtypes(include=[np.number]).columns
for col in numerical_cols:
    if df_processed[col].isnull().sum() > 0:
        median_val = df_processed[col].median()
        df_processed[col].fillna(median_val, inplace=True)
        print(f"  ✓ Filled {col} with median: {median_val}")

print("\nMissing values after imputation:")
print(df_processed.isnull().sum().sum(), "remaining")

# Handle negative Age values (data quality issue mentioned in problem statement)
print("\nHandling negative Age values...")
print(f"  Negative ages found: {(df_processed['Age'] < 0).sum()}")
# Replace negative ages with absolute value
df_processed['Age'] = df_processed['Age'].abs()
print(f"  ✓ Converted negative ages to positive values")

# One-hot encode categorical variables: Diet_Type and Blood_Group
print("\nEncoding categorical variables...")
df_processed = pd.get_dummies(df_processed, columns=['Diet_Type', 'Blood_Group'], drop_first=False)
print(f"  ✓ Diet_Type encoded (Vegetarian, Non-Vegetarian, Vegan)")
print(f"  ✓ Blood_Group encoded (A, B, AB, O)")

print(f"\nDataset shape after preprocessing: {df_processed.shape}")
print(f"Total features: {df_processed.shape[1] - 1} (excluding Target)")

## 3. Train-Test Split & Decision Tree Model

In [ ]:
# ============================================================================
# STEP 3: PREPARE FEATURES & TARGET, THEN SPLIT DATA
# ============================================================================
# Why: We need to separate features (X) from target (y) and split into train/test
# to evaluate model performance on unseen data

from sklearn.tree import DecisionTreeClassifier, plot_tree

print("=" * 70)
print("PREPARING DATA FOR DECISION TREE")
print("=" * 70)

# Separate features (X) and target (y)
X = df_processed.drop('Target', axis=1)  # All features except Target
y = df_processed['Target']                # Target variable (0=Healthy, 1=Unhealthy)

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Feature names: {list(X.columns[:5])}... (showing first 5)")

# Split data: 80% training, 20% testing
# Why 80/20: Standard ratio for good model training while keeping test set large enough
# stratify=y: Ensures both sets have similar class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,           # 20% for testing
    random_state=42,         # For reproducibility
    stratify=y               # Maintains class balance in both sets
)

print(f"\n✓ Training set size: {X_train.shape[0]} samples")
print(f"✓ Testing set size: {X_test.shape[0]} samples")
print(f"✓ Training target distribution:\n{y_train.value_counts()}")
print(f"✓ Testing target distribution:\n{y_test.value_counts()}")

# ============================================================================
# STEP 4: TRAIN DECISION TREE CLASSIFIER
# ============================================================================
# Why Decision Tree:
# - Interpretable: Can visualize decision rules
# - Handles mixed feature types (no scaling needed)
# - Non-linear relationships captured
# - Good for insurance classification tasks

print("\n" + "=" * 70)
print("TRAINING DECISION TREE CLASSIFIER")
print("=" * 70)

# Create and train the decision tree
dt_model = DecisionTreeClassifier(
    max_depth=5,             # Limit tree depth to prevent overfitting
    min_samples_split=10,    # Minimum samples required to split a node
    min_samples_leaf=5,      # Minimum samples required at leaf node
    random_state=42,         # For reproducibility
    class_weight='balanced'  # Handle class imbalance if present
)

# Train the model on training data
dt_model.fit(X_train, y_train)

print("✓ Decision Tree model trained successfully!")
print(f"✓ Tree depth: {dt_model.get_depth()}")
print(f"✓ Number of leaves: {dt_model.get_n_leaves()}")
print(f"✓ Total nodes: {dt_model.tree_.node_count}")

## 4. Model Evaluation

In [ ]:
# ============================================================================
# STEP 5: EVALUATE MODEL PERFORMANCE
# ============================================================================
# Why: Multiple metrics give a complete picture of model performance

print("=" * 70)
print("MODEL PERFORMANCE EVALUATION")
print("=" * 70)

# Make predictions on both training and test sets
y_train_pred = dt_model.predict(X_train)
y_test_pred = dt_model.predict(X_test)

# Get prediction probabilities for ROC-AUC
y_train_pred_proba = dt_model.predict_proba(X_train)[:, 1]
y_test_pred_proba = dt_model.predict_proba(X_test)[:, 1]

# Calculate accuracy scores
train_accuracy = dt_model.score(X_train, y_train)
test_accuracy = dt_model.score(X_test, y_test)

print(f"\nAccuracy Scores:")
print(f"  Training Accuracy: {train_accuracy:.4f} ({train_accuracy*100:.2f}%)")
print(f"  Testing Accuracy:  {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

# Calculate ROC-AUC scores (important for imbalanced data)
train_auc = roc_auc_score(y_train, y_train_pred_proba)
test_auc = roc_auc_score(y_test, y_test_pred_proba)

print(f"\nROC-AUC Scores (measure of discrimination ability):")
print(f"  Training ROC-AUC: {train_auc:.4f}")
print(f"  Testing ROC-AUC:  {test_auc:.4f}")

# Detailed classification report for test set
print(f"\n" + "-" * 70)
print("DETAILED CLASSIFICATION REPORT (Test Set)")
print("-" * 70)
print(classification_report(y_test, y_test_pred, 
                          target_names=['Healthy (0)', 'Unhealthy (1)']))

# Confusion Matrix
print("\nConfusion Matrix (Test Set):")
cm = confusion_matrix(y_test, y_test_pred)
print(cm)
print(f"\n  True Negatives (TN):  {cm[0, 0]}  (Correctly classified as Healthy)")
print(f"  False Positives (FP): {cm[0, 1]}  (Healthy classified as Unhealthy)")
print(f"  False Negatives (FN): {cm[1, 0]}  (Unhealthy classified as Healthy) ⚠️ RISK")
print(f"  True Positives (TP):  {cm[1, 1]}  (Correctly classified as Unhealthy)")

## 5. Visualize Decision Tree

In [ ]:
# ============================================================================
# STEP 6: VISUALIZE THE DECISION TREE
# ============================================================================
# Why: Visual representation shows how the model makes decisions
# - Each node shows: feature, threshold, and samples
# - Orange=Mostly Unhealthy, Blue=Mostly Healthy

print("=" * 70)
print("DECISION TREE VISUALIZATION")
print("=" * 70)

plt.figure(figsize=(25, 12))
plot_tree(dt_model, 
          feature_names=X.columns,
          class_names=['Healthy', 'Unhealthy'],
          filled=True,
          rounded=True,
          fontsize=10)
plt.title("Decision Tree for Health Insurance Classification", 
          fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("✓ Tree visualization complete!")
print("\nHow to read the tree:")
print("  - At each node: if feature <= threshold, go LEFT; else go RIGHT")
print("  - Box color: Orange=more unhealthy, Blue=more healthy")
print("  - samples: Number of samples at that node")
print("  - value: [#Healthy, #Unhealthy]")

## 6. Feature Importance Analysis

In [ ]:
# ============================================================================
# STEP 7: FEATURE IMPORTANCE
# ============================================================================
# Why: Shows which health indicators most influence insurance classification
# - Most important features should be focused on for insurance decisions

print("=" * 70)
print("FEATURE IMPORTANCE ANALYSIS")
print("=" * 70)

# Extract feature importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 10 Most Important Features for Classification:")
print(feature_importance.head(10).to_string(index=False))

# Visualize feature importance
plt.figure(figsize=(12, 6))
top_features = feature_importance.head(10)
sns.barplot(data=top_features, x='Importance', y='Feature', palette='viridis')
plt.title('Top 10 Most Important Features for Health Classification', 
          fontsize=14, fontweight='bold')
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.tight_layout()
plt.show()

# Key insights
print("\n" + "=" * 70)
print("KEY INSIGHTS FOR ANOVA INSURANCE")
print("=" * 70)
print("\n✓ These are the KEY health indicators affecting premium pricing:")
for idx, row in feature_importance.head(5).iterrows():
    print(f"  {idx+1}. {row['Feature']}: {row['Importance']*100:.2f}% importance")

print("\n💡 Recommendation: Focus underwriting on these top 5 features")

## 7. Model Summary & Next Steps

In [ ]:
# ============================================================================
# STEP 8: MODEL SUMMARY FOR BUSINESS DECISION
# ============================================================================

print("\n" + "=" * 70)
print("DECISION TREE MODEL - BUSINESS SUMMARY")
print("=" * 70)

print(f"""
📊 MODEL PERFORMANCE:
  • Test Accuracy: {test_accuracy*100:.2f}% of applicants classified correctly
  • Test ROC-AUC: {test_auc:.4f} (measures discrimination ability)
  • Model Type: Decision Tree (interpretable, rule-based)

🏥 CLASSIFICATION ABILITY:
  • Identifies Healthy individuals with {(cm[0,0]/(cm[0,0]+cm[0,1]))*100:.1f}% accuracy
  • Identifies Unhealthy individuals with {(cm[1,1]/(cm[1,0]+cm[1,1]))*100:.1f}% accuracy
  
⚠️  RISK ANALYSIS (False Negatives):
  • {cm[1,0]} Unhealthy applicants misclassified as Healthy
  • This could lead to underpriced premiums - important to monitor

💡 BUSINESS RECOMMENDATIONS:
  1. Use this model to automate initial health classification
  2. Focus underwriting on TOP 5 FEATURES identified above
  3. High-risk cases (low confidence) → manual review recommended
  4. Periodically retrain model with new applicant data

🎯 USE CASE FOR ANOVA INSURANCE:
  • Premium Pricing: Apply higher premiums to classified "Unhealthy"
  • Coverage Eligibility: Can use confidence scores for approval workflows
  • Risk Assessment: Automated flagging of high-risk applicants

📈 NEXT STEPS:
  1. Compare with other models (Random Forest, Gradient Boosting)
  2. Hyperparameter tuning for better accuracy
  3. Deploy model in production for real-time predictions
  4. Monitor performance over time
""")

print("=" * 70)
print("✅ Decision Tree model successfully created and evaluated!")
print("=" * 70)